# Day 1 — From Model to Agent

## Daily project: Smart Research Assistant

This is the classroom master notebook for Day 1. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the **Environment setup** cell directly below first. On Google Colab it clones the repository, installs packages, and asks for your API key. On your own computer it only loads the `.env` file.
- Every section starts with a small setup cell of its own; if the kernel restarts, rerun that cell and continue.
- Run the code cells in order and read the printed output: each cell prints what changed and why.
- Every lesson ends with a short **Checkpoint** (answers are folded under *Show answer*) and a **Recap**.
- Without an API key everything runs in deterministic **mock** mode and spends no credit. Use the instructor-issued OpenRouter key only for the marked live observations.
- Section 1.8 is the day's single hands-on exercise; a commented reference solution follows its check.

### Day 1 contents

1. [Your First Model Call](#day-1-section-1)
2. [Configuring Model Behaviour](#day-1-section-2)
3. [Structured Outputs](#day-1-section-3)
4. [Tool Calling](#day-1-section-4)
5. [Build the Agent Loop Manually](#day-1-section-5)
6. [Represent the Agent Loop with LangGraph](#day-1-section-6)
7. [Day 1 Project — Smart Research Assistant](#day-1-section-7)
8. [Pivotal Exercise: Complete the Manual Agent Loop](#day-1-section-8)

---


In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
    os.chdir(REPO_DIR / "day_01_model_tools_agent")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


<a id="day-1-section-1"></a>

## 1.1 — Your First Model Call

We begin with the smallest useful AI application:

```text
Question  ->  Model  ->  Text response
```

This is **not yet an agent**. Nothing loops, nothing runs on your machine, and nothing
decides what to do next. Today we add each of those layers one at a time — but first we
have to be able to send a single request and read what comes back.


## Before you begin

### Learning outcomes

- Create the `.env` file that every notebook in this course reads, and verify it without
  ever printing your key in full.
- Send one prompt through the classroom route and name each field of the request.
- Read the usage record and explain what a *context window* is.

Architecture reference: [D01](../diagrams/source/day_01.md).

### Expected observation

With no key you stay in MOCK mode and every cell below still prints a result. With a key,
the wording of the answer changes from run to run, but the request fields and the usage
record stay in exactly the same shape.

## Concept briefing

## Why this day exists

A language model is a generator, not an application. It receives a finite context and
predicts a continuation. It does not automatically know your files, execute Python, or
continue working until a goal is complete. An agentic application is created when host
code gives the model a limited set of possible actions, carries state between turns,
executes approved actions, and decides when the run must stop.

Day 1 removes the apparent magic from this process. By the end, students should be able
to point to the exact line that sends a request, the exact data that describes a tool,
the exact function that executes it, and the exact condition that terminates the loop.

## What a model call actually contains

A typical request contains a model identifier, ordered messages, optional tool schemas,
and generation controls. Messages are not merely a chat transcript. Their roles tell the
provider how each piece should be interpreted:

- `system`: standing instructions and boundaries;
- `user`: the current task or supplied information;
- `assistant`: previous model output, including tool requests;
- `tool`: an observation produced by host-executed code.

Two request fields deserve names of their own. `temperature` controls how freely the
model samples its next token: 0 makes it take the most likely continuation every time,
which is what pipelines and tool selection need; higher values trade reproducibility for
variety. `max_tokens` bounds only the generated answer, and on reasoning models it is
spent on private reasoning tokens before any visible output, which is why a small limit
can truncate a JSON response mid-field.

Everything the model can see in one call - system message, every earlier message, tool
descriptions and the answer being generated - must fit inside its **context window**, a
fixed token budget. Nothing carries over between calls: a model is stateless, and the
appearance of memory comes from the application re-sending the conversation each time.
This is also why a growing conversation costs more per turn, and why Day 3 has to manage
history rather than let it accumulate.

The provider serializes this request into a form the model can process. The model sees
tokens representing instructions, messages and tool descriptions. It does not receive a
live Python function. When it appears to "call" a tool, it is generating structured
tokens that name a function and propose arguments. The host application parses those
tokens, validates the arguments, applies policy, calls ordinary code, and returns the
result in another message.

This distinction is load-bearing:

```text
model proposes structured tokens
-> application validates and authorizes
-> Python executes
-> application records the observation
-> model sees the observation on the next call
```

If the model invents a tool name, supplies the wrong type, or requests a prohibited
action, nothing should happen unless the application accepts the request.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Setting up your .env file

Your API key is a password. It must never appear inside a notebook, a screenshot, or a
commit. The course keeps it in one file called `.env` that Python reads at run time.

**Where does it go?** In the **repository root** — the folder that already contains
`README.md` and `.env.example`, not inside `day_01_model_tools_agent/`.

**Windows (PowerShell), from the repository root:**

```powershell
Copy-Item .env.example .env
notepad .env
```

**macOS / Linux, from the repository root:**

```bash
cp .env.example .env
nano .env      # or: open -e .env
```

**The name matters.** The file is called exactly `.env` — a dot, then `env`, and *no*
extension. Windows Explorer likes to save it as `.env.txt`; if the setup cell cannot find
your key, that is almost always why. Turn on *File name extensions* in Explorer and check.

**What goes inside.** Fill in these two lines (leave the rest of the copied file alone):

```dotenv
OPENROUTER_API_KEY=sk-or-...your own issued key...
OPENROUTER_MODEL=openai/gpt-oss-120b
```

**Never commit it.** `.env` is already listed in `.gitignore`. `.env.example` is the
template that *is* committed and contains no secret. Your key has a course-wide lifetime
spending limit — do not share it and do not paste it into a notebook cell.

**On Google Colab** there is no repository folder to edit, so the day notebook starts with
a *Colab bootstrap* cell instead. It asks for the key with `getpass` (which hides your
typing) and stores it in `os.environ` for that runtime only — nothing is written to disk,
and the key disappears when the runtime is recycled:

```python
import os, getpass
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter key: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-oss-120b"
```

If you have no key at all, do nothing: every notebook runs in MOCK mode.

In [ ]:
# --- Verify the .env file without leaking the key ------------------------------
# os.getenv returns None when a variable is missing. We never use os.environ["..."]
# because that raises KeyError and would stop the whole notebook.
api_key = os.getenv("OPENROUTER_API_KEY")
model_id = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

def mask(secret):
    """Print just enough of a secret to recognise it, never the whole thing."""
    if not secret:
        return "(not set)"
    if len(secret) < 12:
        return "(set, but suspiciously short - did you paste the whole key?)"
    return f"{secret[:6]}...{secret[-4:]}"      # e.g. sk-or-...9f2a

print("OPENROUTER_API_KEY :", mask(api_key))
print("OPENROUTER_MODEL   :", model_id)
print("Mode               :", "LIVE" if api_key else "MOCK (every cell below still runs)")

### Step 1 — Build the client, but only if a key exists

Constructing a client (or a provider object) unconditionally is the classic way to make a
notebook crash on its first cell for every student without a key. We build it inside an
`if`, and print which route we ended up on.

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 2 — What is actually in a model call?

A "model call" is an HTTP request carrying a small dictionary. Building that dictionary
*before* sending it makes the moving parts visible: which model, which messages, how long
the answer may be, and how random it may be.

In [ ]:
# The request is ordinary Python data. Nothing is sent yet.
question = "Explain recursion in two sentences for a beginner."

request_payload = {
    "model": model_id,                  # which model should answer
    "messages": [                       # the whole conversation, oldest first
        {"role": "user", "content": question},
    ],
    "max_tokens": 300,                  # upper bound on the ANSWER length
    "temperature": 0,                   # 0 = pick the most likely next word every time
}

print("Fields the provider will receive:")
for field, value in request_payload.items():
    print(f"  {field:12} = {value}")

### Step 3 — Send the request and read the answer

`send_chat` below is the only place in this notebook that touches the network, and it does
so only when a key was found. Otherwise it returns a fixed mock answer, so the lesson still
runs. Any live error is caught and reported in one line instead of ending the class.

In [ ]:
MOCK_ANSWER = (
    "Recursion is when a function solves a problem by calling itself on a smaller "
    "version of the same problem. It needs a base case, otherwise the calls never stop."
)

def send_chat(messages, max_tokens=300, temperature=0):
    """Return (answer_text, usage_dict). Falls back to the mock on any problem."""
    if client is None:
        # MOCK route: deterministic text, zero tokens, zero cost.
        return MOCK_ANSWER, {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            # "reasoning" is an OpenRouter extra: keep the model's private thinking short
            # and out of the reply, because we only asked for two sentences.
            extra_body={"reasoning": {"effort": "low", "exclude": True}},
        )
        usage = response.usage
        return response.choices[0].message.content, {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "cost_usd": getattr(usage, "cost", 0.0) or 0.0,
        }
    except Exception as exc:
        print("Live call failed, using the mock answer instead ->", type(exc).__name__, exc)
        return MOCK_ANSWER, {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}

answer, usage = send_chat(request_payload["messages"])
print("Question :", question)
print("Answer   :", answer)

### Step 4 — Read the usage record

Every live response reports how many tokens went in and came out. Tokens are the billing
unit: roughly 4 characters of English each. In MOCK mode all counters are zero, which is
itself useful — you can tell at a glance that no credit was spent.

In [ ]:
print("Usage reported by the provider:")
for field, value in usage.items():
    print(f"  {field:18} = {value}")

# A rule of thumb you can apply without any API: about 4 characters per token.
estimated_prompt_tokens = len(question) // 4
print()
print("Rough estimate of the prompt size :", estimated_prompt_tokens, "tokens")
print("Answer length in characters       :", len(answer))
print("Did the model run any Python here?: no - it only generated text")

### Step 5 — The context window

The **context window** is the maximum number of tokens a model can look at in one call:
system message + every earlier message + tool descriptions + the answer being generated.
It is a hard limit, not a suggestion.

Two consequences you will meet all week:

- Nothing is remembered between calls. Each request must carry everything the model needs.
- A conversation that grows forever eventually stops fitting, so Day 3 has to *manage* it.

In [ ]:
# Pretend we keep appending turns to one conversation and never trim it.
CONTEXT_WINDOW_TOKENS = 128_000   # typical for the classroom model; check your model card
tokens_per_turn = 220             # a small question plus a short answer

conversation_tokens = 0
for turn in range(1, 6):
    conversation_tokens += tokens_per_turn
    percent = 100 * conversation_tokens / CONTEXT_WINDOW_TOKENS
    print(f"after turn {turn}: {conversation_tokens:>5} tokens  ({percent:.2f}% of the window)")

turns_until_full = CONTEXT_WINDOW_TOKENS // tokens_per_turn
print()
print("Turns before this conversation stops fitting:", turns_until_full)
print("Cost per call also grows, because the WHOLE history is re-sent every time.")

### Step 6 — The other routes (optional, read-only)

The guided notebooks all use OpenRouter. Two alternatives exist and are shown here for
completeness; you need neither to finish Day 1.

- **Ollama** runs a small model on your own machine. Useful later for comparing model
  capability and latency; it needs a reasonably powerful computer.
- **Direct OpenAI API** is for students who already have their own OpenAI project. A
  ChatGPT subscription and API billing are separate things.

Both cells below are deliberately commented out.

In [ ]:
# --- Optional route A: a local model through Ollama ----------------------------
# %pip install -q ollama
# from ollama import chat
# local = chat(model="qwen3:4b", messages=[{"role": "user", "content": question}])
# print(local.message.content)

# --- Optional route B: your own OpenAI account ---------------------------------
# The SDK reads OPENAI_API_KEY from the environment. Put the model id you actually
# have access to in .env, for example:
#     OPENAI_MODEL=<model id from your OpenAI account>
# from openai import OpenAI
# direct_client = OpenAI()                       # reads OPENAI_API_KEY
# direct = direct_client.responses.create(
#     model=os.getenv("OPENAI_MODEL", "<model id from your OpenAI account>"),
#     input=question,
# )
# print(direct.output_text)

print("Both optional routes are commented out on purpose - nothing ran.")

### Try it yourself

`max_tokens` bounds the **answer**, not the question. Predict what happens if we ask the
same question with `max_tokens=12`, then run the cell below and compare.

In [ ]:
# --- Worked solution ---
# Prediction: in LIVE mode the answer is cut off mid-sentence, because max_tokens is a
# hard stop applied while the model is generating - it is NOT a polite request to be
# brief. In MOCK mode nothing is generated, so our fixed string comes back untouched;
# that difference is a good reminder of what a mock can and cannot teach you.

short_answer, short_usage = send_chat(
    [{"role": "user", "content": question}],
    max_tokens=12,
)
print("max_tokens=12 ->", short_answer)
print("completion_tokens:", short_usage["completion_tokens"])
print()
if client is None:
    print("MOCK mode: the limit was accepted but never applied, because no model ran.")
else:
    print("LIVE mode: notice the sentence simply stops. The model was interrupted.")

## Required live observation

Send one bounded prompt through the issued OpenRouter route and save the response plus usage record. If service access fails, inspect the instructor-captured trace and continue in mock mode.


### Checkpoint

**1. Your setup cell prints `MOCK (no OPENROUTER_API_KEY found)` but you are sure you created the file. What are the two most likely causes?**

<details><summary>Show answer</summary>

1. The file is not really called `.env` — Windows saved it as `.env.txt`. Turn on file
   name extensions and rename it.
2. It is in the wrong folder. It must sit in the **repository root**, beside `README.md`
   and `.env.example`, not inside `day_01_model_tools_agent/`.

A third, rarer cause: the line was written as `OPENROUTER_API_KEY = sk-or-...` with spaces
around the `=`, or the key was wrapped in quotes it does not need.

</details>

**2. The model answered a question about recursion. Did it run any Python to do that, and where did the answer's length limit come from?**

<details><summary>Show answer</summary>

No Python ran. The model only produced text, one token at a time — that is all a model
call does. The length limit came from `max_tokens` in the request *we* built, not from
anything the model chose; and the total of everything sent plus everything generated must
fit inside the model's context window.

</details>

### Recap

- **Limitation we saw:** a model call returns free-form text, remembers nothing between
  calls, and cannot act on your machine.
- **Layer we added:** a reproducible request — `.env` for the key, an explicit payload of
  model, messages, `max_tokens` and `temperature`, and a mock fallback so the lesson never
  depends on the network.
- **Evidence it worked:** the masked-key check printed the mode, the answer printed, and
  the usage record showed exactly what was billed (zero in mock mode).

---

### Section 1.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-2"></a>

## 1.2 — Configuring Model Behaviour

In 1.1 we sent one question and accepted whatever text came back. Now we take control of
the two dials the application actually owns:

```text
System instructions + user request  ->  model  ->  better-shaped text
        temperature / max_tokens  ->  how varied and how long
```

By the end you will have seen instructions improve consistency — and seen that they still
do not give you a data structure you can trust.


## Before you begin

### Learning outcomes

- Separate standing instructions (`system`) from the current task (`user`) and see the
  difference in the output.
- Use `temperature` deliberately and explain what value to pick for what job.
- Show that an instruction is not a contract: ask for a dictionary and get prose.

Architecture reference: [D01](../diagrams/source/day_01.md).

### Expected observation

The constrained answer follows the requested three-section format; the unconstrained one
does not. The "give me a dictionary" answer looks close enough to fool a human and still
breaks `json.loads`.

In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — One `ask` helper for the whole notebook

`ask` builds the request dictionary, sends it only when a key exists, and otherwise
returns a deterministic mock reply. It returns **both** the answer and the payload that
was built, so we can look at the settings we actually sent.

In [ ]:
import json

def mock_reply(messages):
    """Deterministic stand-in for the model.

    It reacts to what was asked, so each demo below shows a genuinely different
    behaviour instead of one canned string.
    """
    system_text = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    user_text = " ".join(m["content"] for m in messages if m["role"] == "user").lower()

    # (a) The "give me a dictionary" request: helpful prose wrapped around the data.
    if "dictionary" in user_text or "keys" in user_text:
        return (
            "Sure! Here is the dictionary you asked for:\n\n"
            "```python\n"
            "{\n"
            '    "definition": "An AI agent chooses bounded actions using a model.",\n'
            '    "example": "It asks for a calculator tool.",\n'
            '    "limitation": "Host code must execute and authorise the action."\n'
            "}\n"
            "```\n\n"
            "Let me know if you would like me to expand any of the fields!"
        )

    # (b) The temperature demo asks for a short name.
    if "name for" in user_text:
        return "Study Pilot"

    # (c) A system message that asks for the three named sections.
    if "definition" in system_text and "limitation" in system_text:
        return (
            "Definition: An AI agent is an application that uses a model to choose "
            "bounded actions.\n"
            "Example: It requests a calculator tool and the application runs it.\n"
            "Limitation: The model cannot execute anything itself; host code must."
        )

    # (d) No standing instructions: one shapeless paragraph.
    return (
        "An AI agent is a program that uses a language model to work towards a goal, "
        "often over several steps, sometimes using extra capabilities, and people use "
        "the word for many different systems, which is part of why it is confusing."
    )


def ask(messages, max_tokens=400, temperature=0):
    """Return (answer_text, payload_actually_built)."""
    payload = {
        "model": COURSE_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if client is None:
        return mock_reply(messages), payload
    try:
        response = client.chat.completions.create(
            **payload, extra_body={"reasoning": {"effort": "low", "exclude": True}}
        )
        return response.choices[0].message.content, payload
    except Exception as exc:
        print("Live call failed, falling back to the mock ->", type(exc).__name__, exc)
        return mock_reply(messages), payload

print("ask() ready. Route:", "OpenRouter" if client else "mock")

### Step 2 — A broad request, with no standing instructions

One `user` message and nothing else. Read the answer and ask yourself: could a program
rely on this shape?

In [ ]:
broad_answer, _ = ask([{"role": "user", "content": "Explain an AI agent."}])
print("--- answer to a bare user message ---")
print(broad_answer)
print()
print("Lines in the answer :", len(broad_answer.splitlines()))
print("Named sections      :", sum(word in broad_answer for word in ("Definition", "Example", "Limitation")), "of 3")

### Step 3 — Add a system message

A **system** message carries standing instructions: how to behave for this whole call. A
**user** message carries the current task. Keeping them apart means you can change the
task without rewriting the rules, and vice versa.

In [ ]:
constrained_messages = [
    {
        "role": "system",
        "content": (
            "You teach engineering students who are new to agentic AI. Use plain language. "
            "Answer in exactly three short sections labelled Definition, Example, and "
            "Limitation, each one sentence long."
        ),
    },
    {"role": "user", "content": "Explain an AI agent."},
]

constrained_answer, _ = ask(constrained_messages)
print("--- answer with standing instructions ---")
print(constrained_answer)
print()
print("Named sections      :", sum(word in constrained_answer for word in ("Definition", "Example", "Limitation")), "of 3")

### Step 4 — `temperature`: how much the model is allowed to vary

`temperature` scales the randomness of the next-token choice.

- `0` — always take the most likely token. Best for extraction, classification, tool use
  and anything you want to be reproducible. Every notebook in this course uses `0`.
- `1` — sample more freely. Useful for brainstorming, bad for pipelines.

Watch the value we actually put in the request, and then what comes back.

In [ ]:
question = [{"role": "user", "content": "Give one two-word name for a study-planner agent."}]

cold_answer, cold_payload = ask(question, max_tokens=60, temperature=0)
hot_answer, hot_payload = ask(question, max_tokens=60, temperature=1)

print("temperature field sent (cold):", cold_payload["temperature"])
print("temperature field sent (hot) :", hot_payload["temperature"])
print()
print("temperature=0 ->", cold_answer.strip()[:120])
print("temperature=1 ->", hot_answer.strip()[:120])
print()
if client is None:
    print("MOCK mode: the mock ignores temperature entirely, so both answers are identical.")
    print("That is exactly the point - the setting is a request to the PROVIDER. The mock")
    print("still shows you the field is present and correct in the payload we built.")
else:
    print("LIVE mode: run this cell a few times. At 0 the answer barely moves; at 1 it")
    print("wanders. Same prompt, same model - only the sampling rule changed.")

### Step 5 — Break it: ask for a data structure

Instructions shape *style* well. Now ask for something a program must parse and try to use
the answer directly.

In [ ]:
dict_answer, _ = ask([{"role": "user", "content": (
    "Explain an AI agent as a dictionary with exactly the keys definition, example, "
    "and limitation."
)}])

print("--- what the model returned ---")
print(dict_answer)
print()
print("--- what happens when the application tries to use it ---")
try:
    parsed = json.loads(dict_answer)
    print("Parsed successfully:", parsed)
except json.JSONDecodeError as error:
    print("json.loads FAILED:", error)
    print()
    print("The data is in there, but it is surrounded by a greeting, a Markdown fence and")
    print("a follow-up offer. A human reads past all that; json.loads cannot.")

### Try it yourself

Could you fix this by stripping the Markdown fence with `str.replace`? Predict whether
that is a reliable fix, then run the worked solution.

In [ ]:
# --- Worked solution ---
# A "clean it up afterwards" fix works on the example in front of you and fails on the
# next one. Below we strip the fence and see it parse - then we feed it a slightly
# different, equally plausible reply and watch the same code break.

def strip_fence(text):
    """Remove a leading prose paragraph and a ```python / ``` fence, if present."""
    if "```" not in text:
        return text
    inner = text.split("```")[1]                # take what is between the first two fences
    return inner.replace("python", "", 1).strip()

cleaned = strip_fence(dict_answer)
try:
    print("Cleaned and parsed:", json.loads(cleaned))
except json.JSONDecodeError as error:
    print("Even after cleaning:", error)

# The next equally reasonable reply uses single quotes and a trailing comment.
other_reply = "Here you go:\n\n{'definition': 'x', 'example': 'y', 'limitation': 'z'}  # done"
try:
    print("Second reply parsed:", json.loads(strip_fence(other_reply)))
except json.JSONDecodeError as error:
    print("Second reply FAILED:", error)
    print()
    print("Conclusion: cleaning up text is guesswork. In 1.3 we stop guessing and make the")
    print("provider return JSON that matches a schema, then validate it with Pydantic.")

### Checkpoint

**1. You want a classifier that labels support tickets as `bug`, `question` or `praise`. What temperature do you choose, and where do the label rules belong — system or user message?**

<details><summary>Show answer</summary>

`temperature=0`, because you want the same ticket to get the same label every time and you
are not looking for creativity. The label definitions are standing instructions that apply
to every ticket, so they belong in the **system** message; the ticket text itself is the
**user** message. That split lets you send thousands of tickets without rewriting the rules.

</details>

**2. The three-section instruction worked. Why is that still not enough for a program that needs `definition`, `example` and `limitation` as fields?**

<details><summary>Show answer</summary>

Because "it worked" is an observation about one run, not a guarantee. The model may add a
friendly opening line, rename a heading, merge two sections, or answer in a different
order — all of which a human forgives and `json.loads` does not. An instruction is a
preference; a schema plus validation is a contract, which is what 1.3 adds.

</details>

### Recap

- **Limitation we saw:** a bare user message produces shapeless prose, and even a good
  system message cannot promise a parsable structure.
- **Layer we added:** deliberate configuration — a system/user split, `max_tokens`, and
  `temperature=0` for reproducible work.
- **Evidence it worked:** the constrained answer contained all three named sections while
  the bare one contained none, and `json.loads` still failed on the "dictionary" reply.

---

### Section 1.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-3"></a>

## 1.3 — Structured Outputs

Instructions improved the answer but gave us nothing a program can rely on. Now we build a
real contract:

```text
Question -> model -> JSON that matches a schema -> Pydantic validation -> Python object
```

The schema tells the provider what shape to produce. Pydantic checks what actually arrived.
Both steps are needed, and they check different things.


## Before you begin

### Learning outcomes

- Define a Pydantic contract and read the JSON Schema it generates.
- Convert that schema into the **strict** form providers require, and explain each edit.
- Validate a response, and read a validation error instead of a crash.

Architecture reference: [D02](../diagrams/source/day_01.md).

### Expected observation

Valid data becomes a typed Python object you can use with dot access. Data that is
plausible to a human but outside the field constraints is rejected, loudly and early.

## Concept briefing

## Why structured output matters

Free-form text is useful for people but unreliable for software. A program cannot safely
assume every response contains the same headings, fields or value types. A schema turns
this ambiguity into a contract. Validation does not make the model correct; it makes a
particular class of failure visible.

Consider a confidence field. The sentence "confidence is high" may be understandable to
a person but difficult to compare. A schema can require a number between 0 and 1. If the
model returns `4.5`, validation rejects it instead of quietly sending bad data deeper into
the application.

The correct mental model is:

- schema validity asks whether the response has an acceptable shape;
- factual evaluation asks whether its claims are correct;
- policy asks whether a requested action is permitted.

These are different checks and should not be collapsed into one model prompt.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — Write the contract as a Pydantic model

Each field states a name, a type, and where useful a constraint. This is the single place
where "what a research summary is" is defined for the rest of the application.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

class ResearchSummary(BaseModel):
    topic: str
    summary: str
    key_points: list[str] = Field(min_length=1, max_length=5)   # 1 to 5 bullet points
    confidence: float = Field(ge=0.0, le=1.0)                   # a probability, not a mood

print("Fields in the contract:")
for name, field in ResearchSummary.model_fields.items():
    print(f"  {name:12} {str(field.annotation):24} {field.metadata}")

### Step 2 — Look at the JSON Schema Pydantic generates

`model_json_schema()` turns the class into the machine-readable description we can hand to
a provider. Read it before sending it: two things in here will be rejected by strict mode.

In [ ]:
raw_schema = ResearchSummary.model_json_schema()
print(json.dumps(raw_schema, indent=2))
print()
print("Problem 1: there is no \"additionalProperties\": false, so extra invented fields are allowed.")
print("Problem 2: key_points carries minItems/maxItems, which strict mode does not support.")

### Step 3 — Convert it to the strict form

`"strict": True` promises the provider will **only** emit tokens that fit the schema. In
exchange the provider enforces a restricted schema dialect:

1. every object must declare `"additionalProperties": false`;
2. every object must list **all** of its properties in `required` (optional fields are
   expressed as a union with `null` instead);
3. the counting/measuring keywords (`minItems`, `maxItems`, `minLength`, `minimum`, …) are
   not part of that dialect and must be removed.

Sending a raw Pydantic schema with `strict: True` is the most common structured-output
bug: the provider returns a 400 and the notebook dies. Rule 3 does **not** weaken us —
Pydantic still enforces those limits locally, after the response arrives.

In [ ]:
# Keywords the strict dialect does not accept. Pydantic keeps enforcing them for us.
UNSUPPORTED = {
    "minItems", "maxItems", "minLength", "maxLength", "pattern", "format",
    "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "multipleOf", "default",
}

def make_strict(node):
    """Return a copy of a JSON Schema that satisfies the strict dialect.

    The function walks the whole tree, because nested objects (and objects inside
    arrays, and objects inside $defs) must follow the same three rules.
    """
    if isinstance(node, list):
        return [make_strict(item) for item in node]
    if not isinstance(node, dict):
        return node

    cleaned = {key: make_strict(value) for key, value in node.items() if key not in UNSUPPORTED}

    if cleaned.get("type") == "object":
        cleaned["additionalProperties"] = False          # rule 1
        cleaned["required"] = list(cleaned.get("properties", {}))   # rule 2
    return cleaned

strict_schema = make_strict(raw_schema)
print(json.dumps(strict_schema, indent=2))
print()
print("additionalProperties present:", strict_schema.get("additionalProperties"))
print("required lists every field  :", strict_schema.get("required"))
print("minItems removed            :", "minItems" not in json.dumps(strict_schema))

### Step 4 — Ask for the structured answer

The `response_format` block is where the schema goes. `provider.require_parameters`
tells OpenRouter to route only to providers that actually support structured output
instead of silently ignoring it.

In [ ]:
MOCK_STRUCTURED = json.dumps({
    "topic": "AI agents",
    "summary": "An application that uses a model to choose bounded actions.",
    "key_points": ["The host executes tools", "The loop needs a step limit"],
    "confidence": 0.9,
})

def ask_structured(prompt, max_tokens=600):
    """Return the raw response TEXT (not yet validated)."""
    if client is None:
        return MOCK_STRUCTURED
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=max_tokens,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "research_summary",
                    "strict": True,
                    "schema": strict_schema,      # the CONVERTED schema, not raw_schema
                },
            },
            extra_body={
                "reasoning": {"effort": "low", "exclude": True},
                "provider": {"require_parameters": True},
            },
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("Live call failed, using the mock response ->", type(exc).__name__, exc)
        return MOCK_STRUCTURED

response_text = ask_structured("Explain an AI agent for a beginner with two or three key points.")
print("Raw text returned by the model:")
print(response_text)

### Step 5 — Validate, and explain failures in one line

`model_validate_json` either returns a typed object or raises. We wrap it once so a
failure prints something a beginner can act on — including the single most common live
failure, a truncated answer.

In [ ]:
def validate_or_explain(raw_text):
    """Return a ResearchSummary, or None after printing a readable diagnosis."""
    try:
        return ResearchSummary.model_validate_json(raw_text)
    except ValidationError as error:
        print("The response did NOT satisfy the contract:")
        print(error)
        if not raw_text.rstrip().endswith("}"):
            print()
            print("HINT: the text does not even end with '}' - it was cut off, not wrong.")
            print("      max_tokens counts the model's private reasoning tokens too, so a")
            print("      small max_tokens plus reasoning can end a response mid-JSON.")
            print("      Fix: raise max_tokens, or lower the reasoning effort.")
        return None

result = validate_or_explain(response_text)
print()
if result is not None:
    print("Validation passed. We now hold a typed object, not a string:")
    print("  type            :", type(result).__name__)
    print("  result.topic    :", result.topic)
    print("  result.confidence:", result.confidence)
    for number, point in enumerate(result.key_points, start=1):
        print(f"  key point {number}    : {point}")

### Step 6 — Break it: plausible data that is still invalid

Nothing about this JSON looks alarming to a person. `confidence: 4.5` would flow straight
into a dashboard, a threshold check, or a database column. Validation stops it here.

In [ ]:
invalid_data = """{
  "topic": "AI agents",
  "summary": "A short summary",
  "key_points": ["Uses a model"],
  "confidence": 4.5
}"""

print("--- confidence out of range ---")
validate_or_explain(invalid_data)

truncated = '{"topic": "AI agents", "summary": "An application that us'
print()
print("--- a truncated response ---")
validate_or_explain(truncated)

### Try it yourself

Add a fourth field `difficulty` that must be an integer from 1 to 5, then check that the
strict conversion still removes the range keywords while Pydantic still rejects a 9.

In [ ]:
# --- Worked solution ---
class EngineeringConcept(BaseModel):
    name: str
    explanation: str
    applications: list[str] = Field(min_length=1, max_length=3)
    difficulty: int = Field(ge=1, le=5)          # 1 = first year, 5 = research level

concept_schema = make_strict(EngineeringConcept.model_json_schema())
print("Strict schema keys for difficulty:", concept_schema["properties"]["difficulty"])
print("(ge/le became minimum/maximum in the raw schema and were then removed -")
print(" the provider does not enforce them, Pydantic does.)")
print()

# Shape is fine, value is not: exactly the case a schema alone would let through.
bad = '{"name": "Recursion", "explanation": "A function calling itself", ' \
      '"applications": ["Tree traversal"], "difficulty": 9}'
try:
    EngineeringConcept.model_validate_json(bad)
except ValidationError as error:
    print("Rejected by Pydantic, as intended:")
    print(error)

good = '{"name": "Recursion", "explanation": "A function calling itself", ' \
       '"applications": ["Tree traversal"], "difficulty": 2}'
print()
print("Accepted:", EngineeringConcept.model_validate_json(good))

### Checkpoint

**1. You send `strict: True` with the raw Pydantic schema and the provider answers with a 400 error. Name two edits that fix it.**

<details><summary>Show answer</summary>

Add `"additionalProperties": false` to every object and list every property in `required`;
and delete the keywords the strict dialect does not accept, such as `minItems`, `maxItems`
and `minLength`. That is exactly what `make_strict` does — and it must walk nested objects
too, not just the top level.

</details>

**2. Validation passed. Does that mean the summary is true?**

<details><summary>Show answer</summary>

No. Three different questions are easy to confuse:

- *shape* — does the response have the right fields and types? (schema + Pydantic)
- *truth* — are the claims correct? (evaluation, Day 2 onwards)
- *policy* — is this action allowed at all? (guardrails, Day 3)

A confidence of `0.9` is a valid float. It is not evidence of anything.

</details>

### Recap

- **Limitation we saw:** free-form text, and even a raw Pydantic schema, cannot be handed
  straight to a strict-mode provider or to `json.loads`.
- **Layer we added:** a schema converted to the strict dialect, plus Pydantic validation
  with a readable failure path including a truncation hint.
- **Evidence it worked:** the structured response validated into a typed object with dot
  access, while `confidence: 4.5` and a cut-off response were both rejected with an
  explanation.

---

### Section 1.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-4"></a>

## 1.4 — Tool Calling

A model generates text. It cannot open your files, call an API, or run your Python. What
it *can* do is emit a structured request saying "I would like the calculator, with these
arguments". Your application decides what happens next:

```text
User -> model REQUESTS a tool -> Python validates -> Python executes -> result goes back to the model
```

That arrow in the middle is the whole lesson. Nothing runs until your code runs it.


## Before you begin

### Learning outcomes

- Describe a Python function to a model with a JSON tool schema.
- Read a `tool_calls` request and see that it is only a request.
- Validate the arguments, execute the function, and feed the observation back.

Architecture reference: [D03](../diagrams/source/day_01.md).

### Expected observation

The model returns a tool name and an arguments string. The calculator runs only after our
own validation step, and an invalid argument leaves the function completely untouched.

In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — Build a calculator that is safe to expose

Never call `eval()` on text a model produced. Below is a **simplified teaching version**:
it parses the expression into a syntax tree and walks it, allowing numbers and four
operators and nothing else. It is short enough to read in one sitting, which is its job.

The version the Day 1 project actually ships is
`src/research_agent/tools.py::calculate` — same idea, more operators, an exponent guard,
and unit tests. We compare the two in Step 2.

In [ ]:
import ast, json, operator

# --- Simplified teaching version -----------------------------------------------
BINARY = {ast.Add: operator.add, ast.Sub: operator.sub,
          ast.Mult: operator.mul, ast.Div: operator.truediv}
UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def evaluate_node(node):
    """Walk one node of the parsed expression. Anything unexpected is refused."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)                       # a plain number
    if isinstance(node, ast.BinOp) and type(node.op) in BINARY:
        return BINARY[type(node.op)](evaluate_node(node.left), evaluate_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY:
        return UNARY[type(node.op)](evaluate_node(node.operand))
    raise ValueError("Only basic arithmetic is allowed")

def teaching_calculator(expression: str) -> str:
    value = evaluate_node(ast.parse(expression, mode="eval").body)
    return str(int(value)) if value.is_integer() else str(value)

print("12 * 7        ->", teaching_calculator("12 * 7"))
print("(3 + 4) / 2   ->", teaching_calculator("(3 + 4) / 2"))

for dangerous in ["__import__('os').getcwd()", "open('secrets.txt').read()"]:
    try:
        teaching_calculator(dangerous)
    except (ValueError, SyntaxError) as error:
        print(f"{dangerous[:26]:28} -> refused: {error}")

### Step 2 — Compare it with the project version

Two differences worth noticing: the shipped version supports more operators, and it caps
the exponent so `2 ** 999999` cannot freeze the kernel. Reading the difference is how you
learn what "hardened" means in practice.

In [ ]:
# PROJECT_ROOT/src is already on sys.path thanks to the course setup cell.
from research_agent.tools import calculate as project_calculator

cases = ["12 * 7", "2 ** 8", "2 ** 999", "__import__('os').getcwd()"]
print(f"{'expression':28} {'teaching version':24} project version")
for expression in cases:
    try:
        teaching = teaching_calculator(expression)
    except Exception as error:
        teaching = f"refused ({type(error).__name__})"
    try:
        project = project_calculator(expression)
    except Exception as error:
        project = f"refused ({type(error).__name__})"
    print(f"{expression:28} {teaching:24} {project}")

print()
print("From here on, use research_agent.tools.calculate. The teaching version stays")
print("in this notebook only so you can read every line of it.")

### Step 3 — Describe the tool to the model

The model never sees your function. It sees this dictionary: a name, a sentence of
description, and a JSON Schema for the arguments. Write the description as if for a
colleague — it is the only thing that tells the model *when* to reach for the tool.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class CalculatorArguments(BaseModel):
    """Our own second line of defence: what we accept, regardless of what was sent."""
    expression: str = Field(min_length=1, max_length=100)

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression instead of calculating mentally.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
            "additionalProperties": False,      # no invented extra arguments
        },
    },
}

print(json.dumps(calculator_tool, indent=2))

### Step 4 — Ask the model, and inspect what comes back

We normalise both routes to plain dictionaries so the rest of the notebook reads the same
whether you are live or mocked.

In [ ]:
MOCK_TOOL_REQUEST = {
    "role": "assistant",
    "content": "",
    "tool_calls": [{
        "id": "mock-call-1",
        "type": "function",
        "function": {"name": "calculator", "arguments": '{"expression": "12 * 7"}'},
    }],
}

messages = [{"role": "user", "content": "What is 12 * 7? Use the calculator."}]

def request_tool_call(messages):
    """Return the assistant message as a plain dict, live or mocked."""
    if client is None:
        return MOCK_TOOL_REQUEST
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=messages,
            tools=[calculator_tool],
            temperature=0,
            max_tokens=400,
            extra_body={"reasoning": {"effort": "low", "exclude": False},
                        "provider": {"require_parameters": True}},
        )
        return response.choices[0].message.model_dump(exclude_none=True)
    except Exception as exc:
        print("Live call failed, using the mock request ->", type(exc).__name__, exc)
        return MOCK_TOOL_REQUEST

assistant_message = request_tool_call(messages)
tool_calls = assistant_message.get("tool_calls") or []

print("Assistant text content :", repr(assistant_message.get("content")))
print("Number of tool requests:", len(tool_calls))
for call in tool_calls:
    print("  id       :", call["id"])
    print("  tool name:", call["function"]["name"])
    print("  arguments:", call["function"]["arguments"], "  <- note: a STRING, not a dict")
print()
print("Has anything been calculated yet? No. This is a request, not a result.")

### Step 5 — Validate the arguments, then execute

Two checks before any Python runs: is this a tool we actually offer, and are the arguments
acceptable? Only then do we call the function.

In [ ]:
AVAILABLE_TOOLS = {"calculator": project_calculator}

def run_tool_call(call):
    """Validate and execute one tool request. Returns the observation text."""
    name = call["function"]["name"]
    if name not in AVAILABLE_TOOLS:
        return f"Tool error: unknown tool '{name}'"          # the model invented a name
    try:
        raw_arguments = json.loads(call["function"]["arguments"])   # string -> dict
        arguments = CalculatorArguments.model_validate(raw_arguments)
    except (json.JSONDecodeError, ValidationError) as error:
        return f"Tool error: bad arguments ({error})"
    try:
        return AVAILABLE_TOOLS[name](arguments.expression)
    except Exception as error:
        return f"Tool error: {error}"

call = tool_calls[0]
observation = run_tool_call(call)
print("tool  :", call["function"]["name"])
print("input :", call["function"]["arguments"])
print("output:", observation)

### Step 6 — Return the observation to the model

The conversation must now contain three things in order: our question, the assistant's
tool request, and a `tool` message carrying the result. The `tool_call_id` is what links
the answer to the question — leave it out and the provider rejects the request.

In [ ]:
messages.append(assistant_message)                       # what the model asked for
messages.append({                                        # what our code observed
    "role": "tool",
    "tool_call_id": call["id"],
    "content": observation,
})

print("Conversation now has", len(messages), "messages:")
for index, message in enumerate(messages):
    requested = [c["function"]["name"] for c in (message.get("tool_calls") or [])]
    print(f"  {index}. role={message['role']:9} tool_requests={requested} content={str(message.get('content'))[:60]!r}")

def final_answer(messages):
    if client is None:
        return f"The calculator returned {observation}, so 12 * 7 = {observation}."
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL, messages=messages, tools=[calculator_tool],
            temperature=0, max_tokens=300,
            extra_body={"reasoning": {"effort": "low", "exclude": True}},
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("Live call failed, using the mock answer ->", type(exc).__name__, exc)
        return f"The calculator returned {observation}."

print()
print("Final answer:", final_answer(messages))

### Step 7 — Break it: an argument our schema refuses

A model can send an empty expression, a 500-character one, or Python code. The tool schema
describes the *shape*; our Pydantic model and the calculator itself decide what is
actually allowed. We prove here that the function is never reached.

In [ ]:
executions = {"count": 0}

def counted_calculator(expression):
    executions["count"] += 1                 # increments ONLY if we really execute
    return project_calculator(expression)

AVAILABLE_TOOLS["calculator"] = counted_calculator

bad_requests = [
    {"id": "b1", "function": {"name": "calculator", "arguments": '{"expression": ""}'}},
    {"id": "b2", "function": {"name": "calculator", "arguments": '{"expression": "' + "9" * 150 + '"}'}},
    {"id": "b3", "function": {"name": "calculator", "arguments": '{"wrong_key": "12 * 7"}'}},
    {"id": "b4", "function": {"name": "calculator", "arguments": 'not json at all'}},
    {"id": "b5", "function": {"name": "calculator", "arguments": '{"expression": "__import__(\'os\').getcwd()"}'}},
    {"id": "b6", "function": {"name": "delete_all_files", "arguments": "{}"}},
]

for request in bad_requests:
    # Validation errors are multi-line; squash them so the table stays readable.
    message = " ".join(run_tool_call(request).split())
    print(f"{request['id']}: {message[:105]}")

print()
print("Times the calculator function actually ran:", executions["count"])
print("Only b5 reached it (its shape was valid) and the AST evaluator refused the code.")
AVAILABLE_TOOLS["calculator"] = project_calculator     # restore

### Checkpoint

**1. The model returns `tool_calls` naming `delete_all_files`. What happens?**

<details><summary>Show answer</summary>

Nothing — unless your code chooses to make it happen. A tool call is generated text. In
Step 5 the lookup `if name not in AVAILABLE_TOOLS` returns a `Tool error: unknown tool`
observation, and the model reads that on its next turn. This is why the registry, not the
model, is the security boundary.

</details>

**2. Why validate arguments with Pydantic when the tool schema already said `expression` is a string?**

<details><summary>Show answer</summary>

The schema is a description sent to the provider; it is advice, and outside strict mode a
model can and does deviate. Even inside strict mode "a string" says nothing about length or
content. The Pydantic model runs on **your** machine, on the data that actually arrived,
and it is the last thing between a model's suggestion and your function. Test `b3` and `b4`
above show requests that never even parse.

</details>

### Recap

- **Limitation we saw:** a model cannot execute anything, so on its own it can only guess
  at arithmetic and outside facts.
- **Layer we added:** a tool schema, a request/validate/execute boundary, and a `tool`
  message that carries the observation back into the conversation.
- **Evidence it worked:** `12 * 7` was answered from a real calculation, and six malformed
  requests produced error observations while the counter proved the function ran once.

---

### Section 1.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-5"></a>

## 1.5 — Build the Agent Loop Manually

One hardcoded tool interaction cannot handle a question that needs two tools, or none, or
an unknown number. The mechanism that makes an application *agentic* is a loop:

```text
model -> decide -> tool -> observation -> model -> ... -> final answer
```

We are going to write that loop by hand, one line at a time, watch the message list grow,
and make it hit its own step limit. Only at the very end do we open the packaged version
in `src/` and confirm it is the same twenty-odd lines.


## Before you begin

### Learning outcomes

- Write the model/tool/observation loop yourself, with a step counter and a stop condition.
- Read the message list after every turn and say why each message is there.
- Prove the limit belongs to your code by making the loop stop at `max_steps`.

Architecture reference: [D04](../diagrams/source/day_01.md).

### Expected observation

Each turn appends one `assistant` message and one `tool` message per executed tool. With
`max_steps=1` the run ends unfinished and says so, instead of looping forever.

## Concept briefing

## Why the application owns termination

After one tool result, the model may ask for another tool or return a final answer. That
creates a loop whose length is not known in advance. It is tempting to write "stop when
finished" in the system message and trust the model. That is not an execution limit. A
confused model can repeat the same request, alternate between tools, or continue refining
an already adequate answer. Each turn consumes time, tokens and money.

Host code therefore enforces a maximum number of steps. Reaching the limit is not the
same as crashing. A good runtime returns a visible status such as `max_steps` together
with the partial trace. Reporting incomplete work honestly is safer than pretending the
run completed.

## Error compounding

Multi-step systems amplify small error rates. Suppose, only for illustration, that each
model decision has a 95% chance of being acceptable and that errors are independent. The
chance that ten decisions are all acceptable is:

```text
0.95 ^ 10 = approximately 0.60
```

The independence assumption is simplistic, but the lesson is useful: a system with many
model decisions can be much less reliable than any single impressive response suggests.
This motivates bounded loops, deterministic validation, fewer calls, clear tools and
evaluation of complete trajectories rather than isolated answers.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Pick a provider and look at the tools

`MockModelProvider` follows the same interface as the real one: give it messages and tool
definitions, get back a `ModelTurn` with either text or tool requests. That is why the
identical loop works in both modes.

In [ ]:
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

# Never construct the live provider unless a key exists - it raises without one.
provider = OpenRouterProvider() if LIVE else MockModelProvider()
print("Provider:", type(provider).__name__)

tools = default_tool_registry()
tool_definitions = [tool.definition for tool in tools.values()]
print()
print("Tools this agent may request:")
for name, tool in tools.items():
    print(f"  {name:20} {tool.definition.description}")

### Step 2 — Seed the conversation

Two messages start every run: the standing instructions, and the question. Everything else
in the list will be produced by the loop.

In [ ]:
from research_agent.agent import SYSTEM_MESSAGE

question = "Explain an AI agent using the local notes and calculate 12 * 7."

messages = [
    Message(role="system", content=SYSTEM_MESSAGE),
    Message(role="user", content=question),
]

print("SYSTEM_MESSAGE (what the model is told once, up front):")
print(SYSTEM_MESSAGE)
print("Messages before the loop starts:", len(messages))

### Step 3 — Do one turn by hand

Call the model. It answers with **either** text (it is finished) **or** tool requests (it
wants something done first). Deciding which of those happened is the branch the whole loop
is built around.

In [ ]:
turn = provider.complete(messages, tool_definitions)

print("turn.content    :", repr(turn.content))
print("turn.tool_calls :", [(call.name, call.arguments) for call in turn.tool_calls])
print()
print("Finished?", "yes - this is the final answer" if not turn.tool_calls else "no - it wants a tool first")

# Whatever it said, it becomes part of the conversation.
messages.append(Message(role="assistant", content=turn.content, tool_calls=turn.tool_calls))
print("Messages now:", len(messages))

### Step 4 — Execute the request and append the observation

`tool.execute` validates the arguments and returns a `ToolResult` instead of raising, so a
bad request becomes an observation the model can read and recover from. Note the
`tool_call_id`: it is what ties the answer to the question.

In [ ]:
for call in turn.tool_calls:
    tool = tools.get(call.name)
    if tool is None:
        output = f"Tool error: unknown tool '{call.name}'"
    else:
        output = tool.execute(call.id, call.arguments).output
    print(f"executed {call.name}{call.arguments} -> {output[:80]}")
    messages.append(
        Message(role="tool", name=call.name, tool_call_id=call.id, content=output)
    )

print()
print("Messages now:", len(messages))
for index, message in enumerate(messages):
    print(f"  {index}. {message.role}")

### Step 5 — Now write the whole loop

Steps 3 and 4 repeat until the model stops asking for tools. Three things make it an
agent loop rather than a `while True`:

1. a **step counter** with a hard maximum;
2. a **branch** on `turn.tool_calls` — text means finished;
3. **validation** of the final answer before we call the run a success.

Read the loop, then read the printed trace under it.

In [ ]:
from pydantic import ValidationError
from research_agent.schemas import ResearchResponse

def run_agent(question, max_steps=5, show_messages=True):
    """The whole agent, in about 25 lines."""
    messages = [Message(role="system", content=SYSTEM_MESSAGE),
                Message(role="user", content=question)]

    for step in range(1, max_steps + 1):                      # 1. bounded, never while True
        turn = provider.complete(messages, tool_definitions)
        messages.append(Message(role="assistant", content=turn.content,
                                tool_calls=turn.tool_calls))

        if not turn.tool_calls:                               # 2. no tools = final answer
            try:
                answer = ResearchResponse.model_validate_json(turn.content)
            except ValidationError as error:                  # 3. validate before trusting
                return {"status": "failed", "steps": step, "messages": messages,
                        "error": f"Final response failed validation: {error}"}
            return {"status": "completed", "steps": step, "messages": messages,
                    "answer": answer}

        for call in turn.tool_calls:                          # execute what was requested
            tool = tools.get(call.name)
            output = (tool.execute(call.id, call.arguments).output if tool
                      else f"Tool error: unknown tool '{call.name}'")
            messages.append(Message(role="tool", name=call.name,
                                    tool_call_id=call.id, content=output))

        if show_messages:
            print(f"--- after model turn {step}: {len(messages)} messages ---")
            for index, message in enumerate(messages):
                requested = [c.name for c in message.tool_calls]
                print(f"   {index}. role={message.role:9} tool_requests={requested}")

    return {"status": "max_steps", "steps": max_steps, "messages": messages,
            "error": f"Agent stopped after {max_steps} steps"}

result = run_agent(question)
print()
print("status:", result["status"], "| model turns:", result["steps"])

### Step 6 — Read the result

The loop ended because the model returned text instead of a tool request, and that text
passed the schema. Both facts are checked by our code, not asserted by the model.

In [ ]:
answer = result.get("answer")
if answer is not None:
    print(answer.model_dump_json(indent=2))
    print()
    print("Tools the run actually used:", answer.tools_used)
else:
    print("No validated answer. error:", result["error"])

print()
print("Observations the model saw:")
for message in result["messages"]:
    if message.role == "tool":
        print(f"  {message.name:20} -> {message.content[:70]}")

### Step 7 — Force the limit

Give the same two-tool question a budget of one turn. The model cannot possibly finish,
and the honest outcome is `max_steps` plus a partial trace — not a crash, and not a
pretend answer.

In [ ]:
limited = run_agent(question, max_steps=1, show_messages=False)

print("status :", limited["status"])
print("steps  :", limited["steps"])
print("error  :", limited.get("error"))
print("messages produced:", len(limited["messages"]))
print()
print("Compare with the unlimited run:", result["status"], "in", result["steps"], "turns.")
print("The step limit lives in OUR for-loop. No instruction to the model can enforce it.")

### Step 8 — The same loop, packaged

`AgentRunner` in `src/research_agent/agent.py` is the loop you just wrote, plus two extras:
it records token usage, and it refuses to run the *same* tool with the *same* arguments
twice (a stuck model would otherwise burn the whole budget). Open the file and match it
line by line against Step 5.

In [ ]:
from research_agent.agent import AgentRunner

runner = AgentRunner(provider=provider, tools=tools, max_steps=5)
packaged = runner.run(question)

print("Our loop      :", result["status"], "in", result["steps"], "turns,",
      len(result["messages"]), "messages")
print("AgentRunner   :", packaged.status, "in", packaged.steps, "turns,",
      len(packaged.messages), "messages")
print("Usage recorded:", packaged.usage.model_dump())
print()
print("Same trace, same stopping rule - the only new thing is the bookkeeping.")

### Try it yourself

What happens if the loop forgets to append the `tool` observation? Predict it, then run the
worked solution.

In [ ]:
# --- Worked solution ---
# Prediction: the model asks for the tool, we run it, and then we throw the answer away.
# On the next turn the conversation looks exactly as it did before, so the model asks for
# the same thing again... and again, until the step limit stops it. The observation is not
# a log line; it is the only way a result gets back into the model's context.

def run_agent_forgetting_observations(question, max_steps=4):
    messages = [Message(role="system", content=SYSTEM_MESSAGE),
                Message(role="user", content=question)]
    for step in range(1, max_steps + 1):
        turn = provider.complete(messages, tool_definitions)
        messages.append(Message(role="assistant", content=turn.content,
                                tool_calls=turn.tool_calls))
        if not turn.tool_calls:
            return {"status": "completed", "steps": step}
        for call in turn.tool_calls:
            tool = tools.get(call.name)
            output = tool.execute(call.id, call.arguments).output if tool else "?"
            print(f"turn {step}: model asked for {call.name}{call.arguments} -> {output[:40]}")
            # (the Message(role="tool", ...) append is deliberately missing)
    return {"status": "max_steps", "steps": max_steps}

broken = run_agent_forgetting_observations(question)
print()
print("Result:", broken)
print("Every turn repeats the same request, and only max_steps ends it.")
print("This is exactly the failure AgentRunner's duplicate-request check catches early.")

### Checkpoint

**1. Where is the stopping rule, and why can it not live in the system prompt?**

<details><summary>Show answer</summary>

It is the `for step in range(1, max_steps + 1)` header in your own Python. A system prompt
is a request to a text generator; a confused or adversarial model can ignore it, and every
extra turn costs time and tokens. Only host code can *guarantee* the loop ends — which is
also why hitting the limit returns the status `max_steps` and the partial trace instead of
raising.

</details>

**2. A two-tool question produced how many messages, and what is each of them for?**

<details><summary>Show answer</summary>

Seven — count them in the Step 8 output. `system` (standing instructions) and `user` (the
question) start the list. Then each of the two tool turns adds an `assistant` message
holding the request and a `tool` message holding the observation, which is four more.
Finally one `assistant` message carries the JSON answer. Nothing is optional: drop the
`tool` messages and the model never learns the results, as the worked solution above
demonstrates.

</details>

### Recap

- **Limitation we saw:** a single request/execute/respond sequence cannot handle an unknown
  number of steps, and nothing in it can stop a model that keeps asking.
- **Layer we added:** a bounded loop we wrote ourselves — step counter, branch on
  `tool_calls`, observation messages, and validation of the final answer.
- **Evidence it worked:** the trace printed after every turn, the two-tool question
  completed and validated, and `max_steps=1` ended the run honestly with a partial trace.

---

### Section 1.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-6"></a>

## 1.6 — Represent the Agent Loop with LangGraph

You have already written the loop, so nothing new happens today conceptually. LangGraph
gives the same mechanism an explicit shape — named nodes, typed state, and edges you can
draw:

```text
START -> model --final answer--> END
           |-- tool request --> tools --> model
           |-- step limit ----> limit --> END
```

LangGraph is not the agent's intelligence and it does not replace the provider call. It
organises execution, which starts to matter on Day 3 when we need branching, interrupts
and checkpoints.


## Before you begin

### Learning outcomes

- Name the four pieces of a graph: state, node, edge, conditional edge.
- Map each line of your Step 5 loop from 1.5 onto one of them.
- Run the graph and read the final state, including which route ended the run.

Architecture reference: [D05](../diagrams/source/day_01.md).

### Expected observation

The compiled graph prints as a small diagram, the run ends with the same validated answer
as the manual loop, and `max_steps=1` exits through the `limit` node instead of `END`.

If `langgraph` is not installed, every cell prints an install hint and skips — nothing
raises, and notebooks 01-05 and 07 are unaffected.

## Concept briefing

## Workflow or agent?

Not every problem needs an agent. Use ordinary code or a deterministic workflow when the
steps and decision rules are known. Use a hybrid workflow when most steps are fixed but
one bounded judgment benefits from a model. Consider an agent when the next action cannot
be fully predetermined, the action set is small, failures are containable, and success
can be evaluated.

Ask:

1. Are the steps known in advance?
2. Can normal code make the decision reliably?
3. Does the model genuinely add judgment rather than decoration?
4. What is the consequence of a wrong action?
5. Is there a strict step and tool boundary?
6. Can we observe and evaluate the result?

If these questions have weak answers, the correct design is often a workflow, not an
agent.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Check that LangGraph is available

Optional dependencies are imported inside a `try`, so a missing package produces a printed
hint instead of a traceback. `GRAPH_AVAILABLE` then guards every later cell.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

try:
    # Imported only to prove the real API is present, not used directly in this cell.
    from langgraph.graph import StateGraph  # noqa: F401
    GRAPH_AVAILABLE = True
    try:
        installed = version("langgraph")
    except PackageNotFoundError:
        installed = "unknown"
    print("langgraph is installed - version", installed)
except ImportError:
    GRAPH_AVAILABLE = False
    print("Optional: run  pip install langgraph  (it is already in requirements.txt)")
    print("to run this notebook's graph cells. Every cell below will skip safely.")
    print("Nothing else in Day 1 needs it.")

### Step 2 — State is application-owned data

A graph node is an ordinary Python function: it receives the current state and returns the
fields it wants to change. Our state is the same information the manual loop kept in local
variables — the message list, a step counter, the limit, the validated answer, and an error.

In [ ]:
from research_agent.agent import SYSTEM_MESSAGE
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

provider = OpenRouterProvider() if LIVE else MockModelProvider()
tools = default_tool_registry()
print("Provider:", type(provider).__name__)

question = "Explain an AI tool using the local notes and calculate 12 * 7."

initial_state = {
    "messages": [Message(role="system", content=SYSTEM_MESSAGE),
                 Message(role="user", content=question)],
    "steps": 0,               # incremented by the model node
    "max_steps": 5,           # read by the routing function
    "final_response": None,   # filled in when a validated answer arrives
    "error": None,            # filled in on validation failure or step limit
}

print()
print("Initial state:")
for key, value in initial_state.items():
    shown = [m.role for m in value] if key == "messages" else value
    print(f"  {key:15} = {shown}")

### Step 3 — Compile the graph and draw it

`build_graph` lives in `src/research_agent/graph.py`. It registers three nodes (`model`,
`tools`, `limit`), one conditional edge out of `model`, and a plain edge from `tools` back
to `model` — the arrow that makes it a loop.

In [ ]:
graph = None
if GRAPH_AVAILABLE:
    from research_agent.graph import build_graph
    graph = build_graph(provider, tools)
    print(graph.get_graph().draw_mermaid())
else:
    print("Skipped: langgraph is not installed.")

### Step 4 — Invoke it and read the final state

`graph.invoke` runs nodes until something routes to `END`. What comes back is the final
state dictionary, not just an answer — which is why a graph is easy to inspect.

In [ ]:
final_state = None
if graph is not None:
    final_state = graph.invoke(initial_state)
    print("steps taken :", final_state["steps"])
    print("error       :", final_state["error"])
    print()
    if final_state["final_response"]:
        print(final_state["final_response"].model_dump_json(indent=2))
    else:
        print("No validated final response.")
else:
    print("Skipped: langgraph is not installed.")

### Step 5 — The route, read from the messages

The message list tells you which path the run took. It is the same trace the manual loop
produced in 1.5, because it is the same loop.

In [ ]:
if final_state is not None:
    for index, message in enumerate(final_state["messages"]):
        requested = [call.name for call in message.tool_calls]
        preview = message.content[:60].replace("\n", " ")
        print(f"{index}. role={message.role:9} tool_requests={requested} content={preview!r}")
else:
    print("Skipped: langgraph is not installed.")

### Step 6 — Take the limit route

Setting `max_steps` to 1 sends the run through the `limit` node instead of `END`. The
graph does not crash; it records why it stopped, exactly as the manual loop did.

In [ ]:
if graph is not None:
    limited_state = graph.invoke({**initial_state, "max_steps": 1, "steps": 0})
    print("steps         :", limited_state["steps"])
    print("error         :", limited_state["error"])
    print("final_response:", limited_state["final_response"])
    print()
    print("Ended through the 'limit' node, so the error field explains the stop.")
else:
    print("Skipped: langgraph is not installed.")

### Step 7 — Manual loop and graph, side by side

| Manual loop (1.5, Step 5)          | Graph (this notebook)             |
|------------------------------------|-----------------------------------|
| `for step in range(...)`            | the `model` node, visited again   |
| `if not turn.tool_calls:`           | the conditional edge out of `model` |
| the tool execution block            | the `tools` node                  |
| `return {"status": "completed"}`    | the edge to `END`                 |
| `return {"status": "max_steps"}`    | the `limit` node, then `END`      |
| local variables                     | the state dictionary              |

Nothing about the architecture changed. What changed is that the control flow is now data
you can print, draw, pause and resume.

### Checkpoint

**1. If LangGraph is not installed, is anything about your Day 1 agent broken?**

<details><summary>Show answer</summary>

No. The agent is the loop in `src/research_agent/agent.py`, which imports nothing optional.
`graph.py` imports LangGraph inside its factory function precisely so notebooks 01-05 and
07 keep working without it. That is the general pattern for an optional dependency: import
it late, set a flag, and let the dependent cells print a hint and skip.

</details>

**2. What did LangGraph change, and what did it not change?**

<details><summary>Show answer</summary>

Changed: the control flow is explicit and inspectable — nodes, a typed state dictionary and
routing you can draw, pause and checkpoint. Not changed: the provider call, the tool
schemas, argument validation, the step limit, and the final-answer contract. Adding a graph
does not make a system more capable or safer; it makes its orchestration easier to see.

</details>

### Recap

- **Limitation we saw:** a hand-written loop is fine at this size, but its control flow
  only exists as local variables, so it cannot be drawn, paused or resumed.
- **Layer we added:** the same loop expressed as state, nodes and conditional edges — with
  a graceful skip when the optional package is missing.
- **Evidence it worked:** the graph printed its own diagram, produced the same validated
  answer as the manual loop, and `max_steps=1` exited through the `limit` node.

---

### Section 1.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-7"></a>

## 1.7 — Day 1 Project — Smart Research Assistant

Everything from today, assembled:

```text
model call -> structured tool request -> validated Python tool -> observation
           -> bounded agent loop -> validated final response
```

A demo that works once is not evidence. We finish by running a small behaviour suite and
then deliberately triggering the three failures this system is designed to survive.


## Before you begin

### Learning outcomes

- Run the finished assistant and read status, steps, tools used and token usage.
- Run a four-case behaviour suite and check the tools against expectations.
- Trigger three failures on purpose and name the layer each one belongs to.

Architecture reference: [D01–D05](../diagrams/source/day_01.md).

### Expected observation

The suite completes in MOCK mode with no API credit spent. The failure demonstrations
print `failed` statuses with specific error messages — never a traceback.

## Concept briefing

## What to carry into Day 2

Day 1 creates a bounded model-and-tool system, but the model still relies on information
inside its request or learned during training. Day 2 introduces external knowledge. The
agent loop remains the same; the new question is how to retrieve the right evidence and
prove the answer used it.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Assemble the project

Mock is the default. It is not a lesser mode for this notebook: the behaviour suite is
deterministic, so everyone gets the same table and can compare answers. Set a key in
`.env` when you want to measure real model behaviour.

In [ ]:
from research_agent.agent import AgentRunner
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.tools import default_tool_registry

# LIVE comes from the course setup cell: True only when OPENROUTER_API_KEY was found.
provider = OpenRouterProvider() if LIVE else MockModelProvider()
tools = default_tool_registry()
runner = AgentRunner(provider, tools, max_steps=5)

print("Provider :", type(provider).__name__)
print("Tools    :", list(tools))
print("max_steps:", runner.max_steps)
print()
print("Mock results show that the APPLICATION works. Only a live run says anything")
print("about model quality - never report mock output as model-quality evidence.")

### Step 2 — Run the assistant once

One question that needs both tools. Read every field of the result: the status, how many
model turns it took, what it used, and what it cost.

In [ ]:
project_result = runner.run(
    "Explain what an AI agent is using local notes, then calculate 12 * 7."
)

print("status      :", project_result.status)
print("model turns :", project_result.steps)
print("usage       :", project_result.usage.model_dump())
print()
if project_result.response:
    print(project_result.response.model_dump_json(indent=2))
else:
    print("error:", project_result.error)

### Step 3 — The behaviour suite

Four cases, chosen so that each exercises a different path: no tool, one calculation, one
lookup, and both. For each one we record what we expected and what happened.

In [ ]:
cases = [
    {"id": "direct",      "question": "Give a brief greeting.",
     "expected_tools": []},
    {"id": "calculation", "question": "Calculate 12 * 7.",
     "expected_tools": ["calculator"]},
    {"id": "knowledge",   "question": "Use local notes to explain an AI tool.",
     "expected_tools": ["search_local_notes"]},
    {"id": "two_tools",   "question": "Use notes to explain an AI agent and calculate 12 * 7.",
     "expected_tools": ["calculator", "search_local_notes"]},
]

records = []
for case in cases:
    result = runner.run(case["question"])
    actual_tools = result.response.tools_used if result.response else []
    records.append({
        "case": case["id"],
        "status": result.status,
        "schema_valid": result.response is not None,
        "expected": case["expected_tools"],
        "actual": actual_tools,
        "tool_check": set(actual_tools) == set(case["expected_tools"]),
        "steps": result.steps,
        "tokens": result.usage.prompt_tokens + result.usage.completion_tokens,
        "cost_usd": round(result.usage.cost_usd, 6),
    })

header = f"{'case':12} {'status':10} {'schema':7} {'tools ok':9} {'steps':6} {'tokens':7} cost"
print(header)
print("-" * len(header))
for row in records:
    print(f"{row['case']:12} {row['status']:10} {str(row['schema_valid']):7} "
          f"{str(row['tool_check']):9} {row['steps']:<6} {row['tokens']:<7} {row['cost_usd']}")

passed = sum(row["tool_check"] and row["schema_valid"] for row in records)
print()
print(f"{passed} of {len(records)} cases behaved as expected.")

### Step 4 — Failure 1: an invalid tool argument

The model asks for the calculator with an argument our contract refuses. A tool returns a
`ToolResult` rather than raising, so the run continues and the model gets to read what went
wrong. Nothing here is a "hallucination" — it is a boundary doing its job.

In [ ]:
bad_arguments = [
    {"expression": ""},                                  # too short for the schema
    {"expression": "9" * 150},                           # too long for the schema
    {"wrong_key": "12 * 7"},                             # the required field is missing
    {"expression": "__import__('os').getcwd()"},         # shape fine, content refused
    {"expression": "1 / 0"},                             # valid arithmetic, invalid maths
]

calculator = tools["calculator"]
for index, arguments in enumerate(bad_arguments, start=1):
    outcome = calculator.execute(f"demo-{index}", arguments)
    shown = str(arguments)[:44]
    # Validation errors span several lines; squash them into one readable line.
    message = " ".join(outcome.output.split())
    print(f"{index}. {shown:46} is_error={outcome.is_error}")
    print(f"   -> {message[:100]}")

print()
print("Every one produced an observation the model can read. None raised, and none of")
print("them reached the arithmetic evaluator with data it had not approved.")

### Step 5 — Failure 2: a repeated tool request, stopped

A stuck model asks for the identical tool call over and over. Each request carries a
*new* id, so the runner cannot compare ids — it compares the tool name plus the arguments.
Without that check the run would quietly spend its whole step budget.

In [ ]:
from research_agent.schemas import Message, ModelTurn, ToolCall, ToolDefinition

class StuckProvider:
    """Always requests the same calculation, with a fresh call id every time."""

    def __init__(self):
        self.calls = 0

    def complete(self, messages: list[Message], tool_definitions: list[ToolDefinition]) -> ModelTurn:
        self.calls += 1
        return ModelTurn(tool_calls=[
            ToolCall(id=f"call-{self.calls}", name="calculator",
                     arguments={"expression": "2 + 2"})
        ])

stuck_provider = StuckProvider()
stuck_result = AgentRunner(stuck_provider, tools, max_steps=5).run("Calculate 2 + 2")

print("status              :", stuck_result.status)
print("error               :", stuck_result.error)
print("model turns used    :", stuck_result.steps, "of a possible 5")
print("provider calls made :", stuck_provider.calls)
print()
print("Stopped on the second identical request instead of burning all five turns.")
print("The signature is (tool name, arguments) - the call id is deliberately excluded,")
print("because a new id on every request would make every signature look unique.")

### Step 6 — Failure 3: invalid structured output

The last thing a model produces must satisfy `ResearchResponse`. Here the model answers in
friendly prose, which is exactly the failure mode 1.2 warned about. The run ends `failed`
with a specific message, and the partial trace is still available for debugging.

In [ ]:
class ChattyProvider:
    """Answers in prose instead of the JSON contract."""

    def complete(self, messages: list[Message], tool_definitions: list[ToolDefinition]) -> ModelTurn:
        return ModelTurn(content="Sure! An AI agent is a program that uses tools. Hope that helps!")

chatty_result = AgentRunner(ChattyProvider(), tools, max_steps=5).run("Explain an AI agent")

print("status  :", chatty_result.status)
print("response:", chatty_result.response)
print()
print("error (first 3 lines):")
for line in (chatty_result.error or "").splitlines()[:3]:
    print("   ", line)
print()
print("Messages kept for debugging:", [m.role for m in chatty_result.messages])

### Step 7 — Name the failing layer

When something goes wrong, resist the word "hallucination" and locate the layer instead.
Each row below maps to something you have now seen printed.

| What you observe          | Failing layer               | Where to look                        |
|---------------------------|-----------------------------|--------------------------------------|
| Wrong tool chosen         | model selection / prompting | the tool `description` text          |
| `Tool error: bad arguments` | model–schema boundary     | the argument model in `tools.py`     |
| `Tool error: <exception>` | Python execution            | the tool function itself             |
| `Final response failed validation` | output contract    | `ResearchResponse` and the system prompt |
| `Duplicate tool request stopped` | model looping        | tool descriptions, or the step budget |
| `max_steps`               | termination / control       | `AgentRunner(max_steps=...)`         |

### Try it yourself

Add a fifth behaviour case that you expect to *fail* the tool check, and confirm the suite
reports it rather than hiding it.

In [ ]:
# --- Worked solution ---
# A test suite is only useful if it can go red. We add a case whose expectation is
# deliberately wrong: the greeting needs no tool, but we claim it should use the
# calculator. A trustworthy suite must report False here.

failing_case = {"id": "expect_fail", "question": "Give a brief greeting.",
                "expected_tools": ["calculator"]}

result = runner.run(failing_case["question"])
actual_tools = result.response.tools_used if result.response else []

print("case          :", failing_case["id"])
print("expected tools:", failing_case["expected_tools"])
print("actual tools  :", actual_tools)
print("tool_check    :", set(actual_tools) == set(failing_case["expected_tools"]))
print("status        :", result.status, "(the RUN succeeded; the EXPECTATION did not)")
print()
print("Note the distinction: a completed run with the wrong tools is still a behaviour")
print("failure. Day 2 turns this idea into a proper golden set.")

### Checkpoint

**1. The suite reports `status=completed` for every case but `tool_check=False` for one of them. Is the project working?**

<details><summary>Show answer</summary>

Partly. `completed` only means the loop finished and the final answer satisfied the schema.
`tool_check` is about behaviour: did it use the tools we expected? A run can be perfectly
well-formed and still answer arithmetic from memory instead of calling the calculator.
Those are separate checks and the table keeps them separate on purpose.

</details>

**2. Why is a repeated tool request treated as a failure rather than just allowed to run again?**

<details><summary>Show answer</summary>

Because the identical call with identical arguments returns the identical observation, so
the model has learned nothing and the conversation is in a loop. Each repetition costs a
full model call. Stopping on the second one gives a clear error naming the tool, while
`max_steps` would eventually stop it too — later, more expensively, and with a vaguer
message.

</details>

### Recap

- **Limitation we saw:** one successful demo says nothing about behaviour, and a run can be
  well-formed and still wrong.
- **Layer we added:** a repeatable behaviour suite plus three deliberate failure
  demonstrations mapped to the layer that catches each one.
- **Evidence it worked:** four cases printed with status, schema validity, tool check,
  steps and cost; bad arguments produced observations instead of exceptions; the repeated
  request stopped after two turns; and prose output was rejected by the contract.

---

### Section 1.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-8"></a>

## 1.8 — Pivotal Exercise: Complete the Manual Agent Loop

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.


## Why this mechanism matters

A tool-using agent is an application-controlled loop. The model proposes either a tool request or a final answer; Python validates, dispatches, records the observation, and decides whether another step is allowed. Day 1.5 built this loop with you; here you write it alone.

## Contract

Implement `run_agent`. Reject unknown tools with `ValueError`, append every tool result to `messages` as `{"role": "tool", "name": ..., "content": ...}`, return the final text, and raise `RuntimeError` when `max_steps` is exhausted.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def run_agent(model, tools, messages, max_steps=4):
    """Run a bounded observe-dispatch-append loop and return final text.

    model(messages) returns either
        {"type": "final", "text": "..."}                       -> return the text
        {"type": "tool", "name": "...", "arguments": {...}}    -> run the tool, append, loop
    """
    # TODO: repeat for at most max_steps
    # TODO: ask model(messages) for the next response
    # TODO: return response["text"] when type == "final"
    # TODO: validate the tool name, run tools[name](**arguments)
    # TODO: append {"role": "tool", "name": ..., "content": ...} to messages
    # TODO: after the loop, raise RuntimeError("step limit reached")
    raise NotImplementedError("Complete the agent loop")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    calls = []
    def add(a, b):
        calls.append((a, b))
        return a + b

    # Case 1: one tool request, then a final answer.
    responses = iter([
        {"type": "tool", "name": "add", "arguments": {"a": 2, "b": 3}},
        {"type": "final", "text": "The result is 5."},
    ])
    history = [{"role": "user", "content": "Add 2 and 3"}]
    assert run_agent(lambda messages: next(responses), {"add": add}, history) == "The result is 5."
    assert calls == [(2, 3)], "the tool must run exactly once with the model's arguments"
    assert any(item.get("role") == "tool" for item in history), "the tool result must be appended"

    # Case 2: an unknown tool must be rejected BEFORE anything runs.
    bad = iter([{"type": "tool", "name": "delete_everything", "arguments": {}}])
    try:
        run_agent(lambda messages: next(bad), {"add": add}, [{"role": "user", "content": "x"}])
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown tool name must raise ValueError")

    # Case 3: a model that never answers must be stopped by the step limit.
    forever = lambda messages: {"type": "tool", "name": "add", "arguments": {"a": 1, "b": 1}}
    try:
        run_agent(forever, {"add": add}, [{"role": "user", "content": "x"}], max_steps=3)
    except RuntimeError:
        pass
    else:
        raise AssertionError("the loop must raise RuntimeError after max_steps")
    print("PASS: dispatch, unknown-tool rejection, and step limit all behave correctly")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def run_agent(model, tools, messages, max_steps=4):
    """Run a bounded observe-dispatch-append loop and return final text."""
    for step in range(1, max_steps + 1):                 # bounded: the loop can never run forever
        response = model(messages)                        # 1. ask the model what to do next
        if response["type"] == "final":                   # 2a. it answered -> we are done
            return response["text"]
        if response["type"] != "tool":                    # anything else is a protocol error
            raise ValueError(f"Unexpected response type: {response['type']!r}")
        name = response["name"]
        if name not in tools:                             # 2b. validate BEFORE executing anything
            raise ValueError(f"Unknown tool requested: {name!r}")
        result = tools[name](**response["arguments"])     # 3. the HOST runs the function, not the model
        messages.append({"role": "tool", "name": name, "content": str(result)})  # 4. record the observation
        print(f"step {step}: ran {name}{response['arguments']} -> {result}")
    raise RuntimeError(f"Stopped after {max_steps} steps without a final answer")  # 5. safe termination

print("Reference run_agent defined. Re-run the check cell above to see PASS.")

## Explain

**Why must the application, rather than the model, own tool execution and termination?**

<details><summary>Show answer</summary>

The model only emits text that <em>looks like</em> a request. Only the host can check the tool exists, validate the arguments, decide whether it is allowed, actually run it, and count steps. If the model owned termination, a confused or malicious prompt could loop forever or call anything.

</details>

**Why is the unknown-tool check placed before <code>tools[name](...)</code> and not after?**

<details><summary>Show answer</summary>

Once a function has run, its side effects have happened. Validation must come first so a bad request is rejected with zero effects.

</details>

---

### Section 1.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 1 completion checklist

- [ ] I can explain how every section contributes to the **Smart Research Assistant**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I attempted the pivotal exercise before reading its reference solution, and I can explain the solution line by line.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
